# 6. Feature Importance Analysis with Integrated Gradients

This notebook uses Integrated Gradients (IG) to identify the key molecular drivers for each of the identified T2D subtypes.

**Key Steps:**
1.  **Load Final Data:** Load the clinical data with the stable subtype labels assigned in the previous notebook.
2.  **Select High-Performance Models:** Filter the trained models to keep only those that meet the performance threshold (e.g., ACC > 0.75) for a robust analysis.
3.  **Calculate Feature Attributions:**
    *   Apply the Integrated Gradients method to calculate attribution scores for all omics features.
    *   Average the scores across the selected models to obtain stable feature importance values.
4.  **Visualize Results:**
    *   Plot the relative contribution of each omics modality for each subtype.
    *   Identify and plot the top contributing features.
5.  **Clinical Complication Analysis:** Analyze the prevalence of diabetic complications across the identified subtypes.

### 6.1. Import Libraries and Configuration


In [1]:
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from glob import glob
import plotly.graph_objects as go
import plotly.express as px
from tqdm import tqdm

# Import Captum for feature attribution
from captum.attr import IntegratedGradients

# Add the project's 'src' directory to the Python path
sys.path.append('../src')

# Import custom modules
import config
from models import MIC
from utils import set_seed, map_clusters_to_subtypes, cluster_accuracy

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


### 6.2. Load Final Data and High-Performance Models


In [2]:
# Load the clinical data with final labels
results_path = config.PROCESSED_DATA_DIR / "final_clinical_data_with_labels.csv"
final_clinical_df = pd.read_csv(results_path)

# Load the processed data, which now includes feature names
data_path = config.PROCESSED_DATA_DIR / "processed_dataset.pt"
processed_data = torch.load(data_path, map_location=device) # Ensure tensors are on the correct device from the start

# Load feature names directly from the saved dataset file
genotype_features = processed_data['genotype_features']
proteome_features = processed_data['proteome_features']
metabolite_features = processed_data['metabolite_features']

# Create a dataset and loader for the entire dataset
eval_dataset = TensorDataset(
    processed_data['input_genotype'],
    processed_data['input_proteome'],
    processed_data['input_metabolite'],
    processed_data['output_clinical']
)
ig_loader = DataLoader(eval_dataset, batch_size=config.BATCH_SIZE)

print("Final data and feature names loaded successfully.")


/tmp/ipykernel_212624/1664229716.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  processed_data = torch.load(data_path, map_location=device) # Ensure tensors are on the 

Final data and feature names loaded successfully.


### 6.3. Integrated Gradients Analysis

To interpret the model, we use Integrated Gradients (IG) to calculate feature attributions. IG identifies the most influential omics features for assigning a sample to its predicted subtype.

First, we will define the output as the soft assignment probability of a sample belonging to its predicted cluster.


In [3]:
# This wrapper class adapts the MIC model for Captum's Integrated Gradients.
class MIC_IG_Wrapper(nn.Module):
    def __init__(self, mic_model):
        super().__init__()
        self.mic_model = mic_model

    def forward(self, genotype, proteome, metabolite):
        z = self.mic_model.encode(genotype, proteome, metabolite)
        q = self.mic_model.soft_cluster_assignment(z)
        return q

# --- Integrated Gradients Analysis Setup ---
subtypes = ["SIRD", "SIDD", "MOD", "MARD"]
subtype_attributions = {subtype: {'genotype': [], 'proteome': [], 'metabolite': []} for subtype in subtypes}
baseline_geno = processed_data['input_genotype'].mean(dim=0, keepdim=True).to(device)
baseline_prot = processed_data['input_proteome'].mean(dim=0, keepdim=True).to(device)
baseline_metab = processed_data['input_metabolite'].mean(dim=0, keepdim=True).to(device)

# Load all model states to iterate through them
all_model_paths = sorted(glob(str(config.MODEL_SAVE_DIR / "*.pth")))
all_model_states = [torch.load(path, map_location=device) for path in all_model_paths]

print("Starting Integrated Gradients analysis with original workflow...")
print(f"Analyzing all {len(all_model_states)} models and filtering internally.")

# Benchmark labels for on-the-fly accuracy calculation
benchmark_labels = final_clinical_df['kmeans_cluster'].values

# --- Main Loop: Replicates the original script's logic with proper seeding ---
input_dims = {
    'genotype': processed_data['input_genotype'].shape[1],
    'proteome': processed_data['input_proteome'].shape[1],
    'metabolite': processed_data['input_metabolite'].shape[1]
}

for run, model_state in enumerate(tqdm(all_model_states, desc="Processing Models")):
    set_seed(100 + run)
    
    # Load the model.
    model = MIC(
        input_dims=input_dims,
        encoder_dims=config.ENCODER_DIMS,
        integration_dims=config.INTEGRATION_DIMS,
        latent_dim=config.LATENT_DIM,
        decoder_dims=config.DECODER_DIMS,
        clinical_output_dim=config.CLINICAL_OUTPUT_DIM,
        cluster_num=config.NUM_CLUSTERS,
        dropout=config.DROPOUT
    ).to(device)
    model.load_state_dict(model_state)
    model.eval()


    temp_df = final_clinical_df.copy()
    pred_labels = model.k_means_clustering(ig_loader, n_init=100, device=device)
    temp_df["mic_cluster"] = pred_labels
    
    temp_df, current_mapping = map_clusters_to_subtypes(temp_df, cluster_col_name='mic_cluster')
    
    if current_mapping is not None:
        accuracy = cluster_accuracy(benchmark_labels, temp_df['mic_cluster'].values)

        if accuracy >= 0.75:
            mic_ig = MIC_IG_Wrapper(model)
            mic_ig.eval()
            ig = IntegratedGradients(mic_ig)
            subtype_to_id_map = {v: k for k, v in current_mapping.items()}

            for subtype_name in subtypes:
                if subtype_name in subtype_to_id_map:
                    target_cluster_id = subtype_to_id_map[subtype_name]
                    sample_indices = temp_df[temp_df['mapped_cluster'] == subtype_name].index.tolist()

                    if not sample_indices: continue
                    
                    # To avoid the warning and improve efficiency, we select only the relevant samples for IG
                    inputs_geno = processed_data['input_genotype'][sample_indices].to(device)
                    inputs_prot = processed_data['input_proteome'][sample_indices].to(device)
                    inputs_metab = processed_data['input_metabolite'][sample_indices].to(device)
                    
                    attributions_geno, attributions_prot, attributions_metab = ig.attribute(
                        inputs=(inputs_geno, inputs_prot, inputs_metab),
                        baselines=(baseline_geno, baseline_prot, baseline_metab),
                        target=target_cluster_id,
                        n_steps=50
                    )
                    
                    mean_attrs_geno = attributions_geno.mean(dim=0).cpu().numpy()
                    mean_attrs_prot = attributions_prot.mean(dim=0).cpu().numpy()
                    mean_attrs_metab = attributions_metab.mean(dim=0).cpu().numpy()
                    
                    subtype_attributions[subtype_name]['genotype'].append(mean_attrs_geno)
                    subtype_attributions[subtype_name]['proteome'].append(mean_attrs_prot)
                    subtype_attributions[subtype_name]['metabolite'].append(mean_attrs_metab)

print("\nIntegrated Gradients analysis complete.")

/tmp/ipykernel_212624/1307775540.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  all_model_states = [torch.load(path, map_location=device) for path in all_model_paths]


Starting Integrated Gradients analysis with original workflow...
Analyzing all 100 models and filtering internally.


Processing Models: 100%|██████████| 100/100 [00:42<00:00,  2.36it/s]


Integrated Gradients analysis complete.


### 6.4. Visualize Omics Contribution

We now visualize the relative contribution of each omics modality to the classification of each subtype, similar to Figure 2b in the paper. We use the sum of positive attribution values to represent the contribution.


In [4]:
# This cell calculates the mean attribution across all models for each subtype,
# and then visualizes the relative contribution of each omics modality.

contribution_data = []

for subtype_name, attributions_dict in subtype_attributions.items():
    # Check if there are any valid attributions for this subtype
    if not attributions_dict['genotype']:
        print(f"Warning: No valid attributions found for subtype {subtype_name}. Skipping.")
        continue
        
    # First, calculate the mean attribution vector across all valid models for each omics type.
    avg_attr_geno = np.mean(attributions_dict['genotype'], axis=0)
    avg_attr_prot = np.mean(attributions_dict['proteome'], axis=0)
    avg_attr_metab = np.mean(attributions_dict['metabolite'], axis=0)
    
    # Sum of positive attributions from the mean vector for each modality
    pos_attr_geno = np.sum(avg_attr_geno[avg_attr_geno > 0])
    pos_attr_prot = np.sum(avg_attr_prot[avg_attr_prot > 0])
    pos_attr_metab = np.sum(avg_attr_metab[avg_attr_metab > 0])
    
    total_pos_attr = pos_attr_geno + pos_attr_prot + pos_attr_metab
    
    # Avoid division by zero if there are no positive attributions
    if total_pos_attr == 0:
        total_pos_attr = 1
        
    # Append data for plotting
    contribution_data.append({'Subtype': subtype_name, 'Omics': 'Genotype', 'Contribution': pos_attr_geno / total_pos_attr * 100})
    contribution_data.append({'Subtype': subtype_name, 'Omics': 'Proteome', 'Contribution': pos_attr_prot / total_pos_attr * 100})
    contribution_data.append({'Subtype': subtype_name, 'Omics': 'Metabolite', 'Contribution': pos_attr_metab / total_pos_attr * 100})


### 6.5. Identify and Visualize Top Features

Here, we identify the top 5 features with the highest mean attribution values for each subtype from each omics modality, similar to Figure 3 in the paper.


In [5]:
# This cell identifies and visualizes the top 5 features with the highest mean attribution values
# for each subtype and omics modality.

# First, create a new dictionary to hold the final mean attributions for easier access.
mean_subtype_attributions = {}

for subtype_name, attributions_dict in subtype_attributions.items():
    if not attributions_dict['genotype']:
        continue
    
    mean_subtype_attributions[subtype_name] = {
        'genotype': np.mean(attributions_dict['genotype'], axis=0),
        'proteome': np.mean(attributions_dict['proteome'], axis=0),
        'metabolite': np.mean(attributions_dict['metabolite'], axis=0)
    }

# Now, find and print the top features from the mean attributions.
for subtype_name, mean_attributions in mean_subtype_attributions.items():
    print(f"--- Top 5 Features for {subtype_name} ---")
    
    # Genotype
    attr_geno = mean_attributions['genotype']
    top_geno_indices = np.argsort(attr_geno)[-5:][::-1] # descending order
    top_geno_features = np.array(genotype_features)[top_geno_indices]
    print(f"Top Genotype Features: {top_geno_features}")

    # Proteome
    attr_prot = mean_attributions['proteome']
    top_prot_indices = np.argsort(attr_prot)[-5:][::-1]
    top_prot_features = np.array(proteome_features)[top_prot_indices]
    print(f"Top Proteome Features: {top_prot_features}")

    # Metabolite
    attr_metab = mean_attributions['metabolite']
    top_metab_indices = np.argsort(attr_metab)[-5:][::-1]
    top_metab_features = np.array(metabolite_features)[top_metab_indices]
    print(f"Top Metabolite Features: {top_metab_features}\n")

# # Example of plotting for one subtype and one omics type
# if 'SIRD' in mean_subtype_attributions:
#     subtype_to_plot = 'SIRD'
#     attr_prot_sird = mean_subtype_attributions[subtype_to_plot]['proteome']
#     top_indices = np.argsort(attr_prot_sird)[-10:] # Top 10 for better viz

#     fig = px.bar(x=np.array(proteome_features)[top_indices], 
#                  y=attr_prot_sird[top_indices],
#                  title=f'Top 10 Proteomic Features for {subtype_to_plot}',
#                  labels={'x': 'Protein', 'y': 'Mean IG Attribution'})
#     fig.show()

--- Top 5 Features for SIRD ---
Top Genotype Features: ['6:20936973_T' '14:24878370_C' '3:185484922_G' '6:166223997_T'
 '19:7968168_G']
Top Proteome Features: ['ephx2' 'myl1' 'ccl15' 'lcp1' 'hsbp1']
Top Metabolite Features: ['L-Homocysteic acid' 'Deoxycholic acid glycine conjugate'
 '6-O-Methyl guanine' 'Glycochenodeoxycholic acid' 'cholesterol-sulfate']

--- Top 5 Features for SIDD ---
Top Genotype Features: ['15:77859960_C' '10:94383821_C' '16:3689678_G' '10:94129202_C'
 '17:6966470_T']
Top Proteome Features: ['acadsb' 'igsf9' 'rab10' 'ephx2' 'ccl18']
Top Metabolite Features: ['GALACTOSAMINE' 'L-Noradrenaline' 'N-Methylnicotinamide' 'LACTOSE'
 'L-Proline']

--- Top 5 Features for MOD ---
Top Genotype Features: ['7:156869648_A' '6:20716085_G' '8:118187232_C' '20:42799719_T'
 '10:122848990_G']
Top Proteome Features: ['siglec8' 'fabp6' 'ryr1' 'pydc1' 'oxct1']
Top Metabolite Features: ['Serotonin' 'AMINOADIPATE' 'isovaleryl-l-carnitine' 'Pyridoxine'
 '3-Hydroxy-octadecenoylcarnitine']

-